# IDAP Alarms — Finding the *Predictive* Ones (Prescriptive already labeled)

**The setup.** Some alarms are **already labeled `PRESCRIPTIVE`** — a person tagged them by hand, each with a note in `what_fixed_it` saying how it was fixed. Those are done; we leave them alone. This notebook only looks at the alarms that **aren't labeled yet**, and asks one question about each:

> Is this alarm **PREDICTIVE** — does it break on a *steady, repeating schedule* we could see coming and get ahead of — **or does the data we have simply not tell us?**

**Being honest up front.** "Predictive" really just means *its timing is regular enough to forecast* — the time between one firing and the next is fairly even, not random. To measure that properly you need the **time of every single firing**. This file only gives us **summary numbers**: a total count, the first and last time it fired, how many days it was active, and how many repair tickets it caused. So we do two separate things:

1. **Squeeze out every bit of signal the summary numbers allow** — with a full set of statistical tests and models — and sort the unlabeled alarms into **PREDICTIVE-CANDIDATE** or **UNCLASSIFIABLE** (with a reason why).
2. **Say clearly where the data runs out.** The last section gives a straight answer on whether the data is good enough, and lists exactly what we'd need (sensor/IoT data, repair-outcome data, the raw event log, …) to turn a *candidate* into a *confirmed* predictive alarm.

**IMPORTANT**: *We have no record of whether past fixes actually worked, so "confidence" in this notebook means a label is **steady** — it doesn't flip when we redraw the lines slightly — **not** that it's been proven right.*

## 0 · Setup & the measures we build

We load the data and turn the raw columns into a few simple measures. Every one is built from **summary numbers only** — we say so plainly, because it's the whole crux of what we can and can't conclude.

- **Active-day ratio** — out of all the days an alarm was "around," on how many did it actually fire? $\;\text{ADR}=\dfrac{\text{distinct active days}}{\text{window length (days)}}\in[0,1]$. Close to 1 = it keeps happening, spread evenly over time (looks forecastable); close to 0 = it all happened in a few days and then went quiet (a burst). *This is the best "regularity" hint the summary numbers can give us.*
- **Events per active day** — on the days it did fire, how many times on average = firings ÷ active days.
- **Mean gap (days)** — rough average time between firings = window ÷ firings (the flip side of the active-day ratio).
- **CM per occurrence** — repair tickets ÷ firings — how much fixing each firing tends to cause.
- **Downtime per occurrence** — total downtime ÷ firings.

**Why these.** They pull apart things the raw counts lump together: the *active-day ratio* separates "spread evenly over time" (what makes a failure forecastable) from "just fires a lot"; *events/day* and *mean gap* describe how intense and how spaced-out it is; *CM per occurrence* measures repair effort fairly no matter how often the alarm fires. What we **don't** have — and can't invent — is the time of each individual firing, which is what you'd need to measure true regularity.

In [ ]:
import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_selection import mutual_info_classif
warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.alpha":.25})
pd.set_option("display.width",200,"display.max_columns",40)

CANDIDATES=[r"/mnt/user-data/uploads/Cur_data.xlsx", "Cur_data.xlsx",
            r"C:\Users\L138397\Downloads\prac\Cur_data.xlsx"]
DATA_PATH=next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
OUT="/mnt/user-data/outputs" if os.path.isdir("/mnt/user-data/outputs") else "."
print("using:", DATA_PATH)